In [1]:
import pandas as pd
print("ready")

ready


In [2]:
!pip install transformers torch datasets


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
from transformers import pipeline
print("transformers ready")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers ready


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print("tokenizer ready")

tokenizer ready


In [5]:
text = "The party shall not be liable for any damages whatsoever."
tokens = tokenizer(text, return_tensors="pt")
print(tokens)

{'input_ids': tensor([[  101,  1996,  2283,  4618,  2025,  2022, 20090,  2005,  2151, 12394,
         18971,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [6]:
tokens_list = tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])
print(tokens_list)

['[CLS]', 'the', 'party', 'shall', 'not', 'be', 'liable', 'for', 'any', 'damages', 'whatsoever', '.', '[SEP]']


In [7]:
from datasets import load_dataset
dataset = load_dataset("theatticusproject/cuad", verification_mode="no_checks")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['pdf'],
        num_rows: 511
    })
})


In [8]:
print(dataset['train'][0])

{'pdf': <pdfplumber.pdf.PDF object at 0x154033620>}


In [9]:
# 提取第一份合同的文字
pdf = dataset['train'][0]['pdf']
first_page = pdf.pages[0]
text = first_page.extract_text()
print(text[:500])

Datasheet for Contract Understanding Atticus Dataset (CUAD)
I.MOTIVATION
A. Who created the dataset (e.g., which team, research group) and on behalf of which entity (e.g. company,
institution, organization)?
The Atticus Project is a non-profit organization whose mission is to harness the power of AI to accelerate
accurate and efficient contract review. The Atticus Project started as a grassroots movement by experienced
lawyers in public companies and leading law firms aiming to achieve high-qual


In [10]:
# 看第5份
pdf = dataset['train'][5]['pdf']
text = ""
for page in pdf.pages[:2]:
    text += page.extract_text() or ""
print(text[:500])

Exhibit 10.8
Affiliate Program / Premium Affiliate Management General
Terms and Conditions
The following General Terms and Conditions are intended for (i) Web site
owners (hereafter, "Affiliates") who wish to participate as Affiliates in
the Affiliate Program provided by element 5 (governed by II. and IV. in
these General Terms and Conditions) on the basis of these General Terms and
Conditions and also for (ii) Software Publishers who distribute their
software products as downloads using the ser


In [11]:
from datasets import load_dataset
dataset = load_dataset("coastalcph/lex_glue", "scotus")
print(dataset)

Generating validation split: 100%|█| 1400/1400 [00:00<00:00, 3394.73 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1400
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1400
    })
})


In [12]:
import pandas as pd

data = {
    "text": [
        "The party shall not be liable for any damages whatsoever.",
        "Either party may terminate this agreement with 30 days notice.",
        "The company reserves the right to modify terms without notice.",
        "All disputes shall be resolved through binding arbitration.",
        "The licensor grants a non-exclusive, royalty-free license.",
        "Client waives all rights to seek damages in any court.",
        "This agreement shall be governed by the laws of California.",
        "The contractor shall indemnify all third party claims.",
        "Payment is due within 30 days of invoice.",
        "The party assumes no responsibility for indirect damages.",
    ],
    "label": [1, 0, 1, 1, 0, 1, 0, 1, 0, 1]
}

df = pd.DataFrame(data)
print(df)

                                                text  label
0  The party shall not be liable for any damages ...      1
1  Either party may terminate this agreement with...      0
2  The company reserves the right to modify terms...      1
3  All disputes shall be resolved through binding...      1
4  The licensor grants a non-exclusive, royalty-f...      0
5  Client waives all rights to seek damages in an...      1
6  This agreement shall be governed by the laws o...      0
7  The contractor shall indemnify all third party...      1
8          Payment is due within 30 days of invoice.      0
9  The party assumes no responsibility for indire...      1


In [13]:
from torch.utils.data import Dataset
import torch

class ContractDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length, return_tensors="pt")
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = ContractDataset(
    df["text"].tolist(),
    df["label"].tolist(),
    tokenizer
)

print(f"Dataset size: {len(train_dataset)}")

Dataset size: 10


In [14]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
print("model ready")

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 4977.88it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

model ready


In [15]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    logging_steps=5,
    save_strategy="no",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()
print("training complete!")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
5,0.678945
10,0.573596
15,0.356287
20,0.207764
25,0.128964
30,0.085823


training complete!


In [17]:
model = model.to("cpu")
model.eval()
correct = 0

for i in range(len(df)):
    inputs = tokenizer(df["text"][i], return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    pred = outputs.logits.argmax().item()
    correct += (pred == df["label"][i])

accuracy = correct / len(df)
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 100.00%


In [18]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

print(f"Train: {len(train_texts)}, Test: {len(test_texts)}")

Train: 8, Test: 2


In [19]:
train_dataset = ContractDataset(train_texts, train_labels, tokenizer)
test_dataset = ContractDataset(test_texts, test_labels, tokenizer)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    logging_steps=5,
    save_strategy="no",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()
print("training complete!")

Step,Training Loss
5,0.111974
10,0.020902
15,0.009858
20,0.006438


training complete!


In [20]:
model = model.to("cpu")
model.eval()
correct = 0

for i in range(len(test_texts)):
    inputs = tokenizer(test_texts[i], return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    pred = outputs.logits.argmax().item()
    print(f"Text: {test_texts[i][:50]}...")
    print(f"Predicted: {pred}, Actual: {test_labels[i]}")
    correct += (pred == test_labels[i])

accuracy = correct / len(test_texts)
print(f"\nAccuracy: {accuracy:.2%}")

Text: Payment is due within 30 days of invoice....
Predicted: 0, Actual: 0
Text: Either party may terminate this agreement with 30 ...
Predicted: 0, Actual: 0

Accuracy: 100.00%
